# Titanic — Clean Solution & Model Guide

A correct, honest pipeline for the Titanic competition. Goals:

1. Build the pipeline **once** (engineer features, transform, tune).
2. Measure generalization with **cross-validation**, not a lucky split.
3. Save a real `../submition/submission.csv`.

**Honest expectation first:** with ~891 rows and ~20% unreliable labels, the
realistic ceiling for this dataset is **~82-85% accuracy**. Scores above that
overfit. 90%+ on a private/public report usually means leakage, not skill.
Good tuning buys a point or two on top of a solid baseline — benchmark everything
with CV before adding complexity (the single decision tree below sets that floor).

In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier,
                              VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV)

In [2]:
def engineer(df):
    return (
        df.copy()
        .assign(Ticket_code=lambda x: x['Ticket'].str.split(' ').str[0])
        .assign(Cabin=lambda x: x['Cabin'].str[0])
        .assign(Sex=lambda x: (x['Sex'] == 'male').astype(int))
        .assign(FamilySize=lambda x: x['SibSp'] + x['Parch'] + 1)
        .assign(IsAlone=lambda x: (x['FamilySize'] == 1).astype(int))
        .drop(columns=['Ticket', 'SibSp', 'Parch'])
    )

train = engineer(pd.read_csv('train.csv').drop(columns=['Survived']))
test = engineer(pd.read_csv('test.csv'))
y = pd.read_csv('train.csv')['Survived']

print('train:', train.shape, '| test:', test.shape)

train: (891, 11) | test: (418, 11)


In [3]:
numerical = ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'IsAlone']
categorical = ['Cabin', 'Ticket_code', 'Embarked']

X = train[numerical + categorical]
X_test = test[numerical + categorical]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler()),
    ]), numerical),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical),
])

def cv_score(model):
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    return scores.mean(), scores.std()

## Baselines

Two cheap models set the "must beat" bar. `LogisticRegression` is the classic
Titanic workhorse; `DecisionTreeClassifier` (not depth-limited) shows raw
trees overfit the noise and are a poor pick here.

In [4]:
lr = Pipeline([('pre', preprocessor),
                  ('clf', LogisticRegression(max_iter=1000, random_state=42))])

mean, std = cv_score(lr)
print(f'LogisticRegression  CV: {mean:.4f} +/- {std:.4f}')

LogisticRegression  CV: 0.8148 +/- 0.0148


In [5]:
dt = Pipeline([('pre', preprocessor),
                  ('clf', DecisionTreeClassifier(random_state=42))])

mean, std = cv_score(dt)
print(f'DecisionTree        CV: {mean:.4f} +/- {std:.4f}')

DecisionTree        CV: 0.8305 +/- 0.0136


## Random Forest — proper grid search

Tune the *real* pipeline inside `GridSearchCV` so preprocessing is re-fit on
every fold (never leak CV folds through the fit step). Report both the CV-mean
used for model selection and the held-out validation accuracy.

In [6]:
rf = Pipeline([('pre', preprocessor),
                  ('clf', RandomForestClassifier(random_state=42))])

param_grid = {
    'clf__n_estimators': [200, 400],
    'clf__max_depth': [None, 10, 15],
    'clf__min_samples_leaf': [1, 2],
    'clf__max_features': [0.3, 0.5, None],
}

grid = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print('Best CV  (train):', round(grid.best_score_, 4))
print('Best params:', grid.best_params_)
print('Validation     :', round(accuracy_score(y_val, grid.predict(X_val)), 4))

Best CV  (train): 0.8344
Best params: {'clf__max_depth': 10, 'clf__max_features': None, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 200}
Validation     : 0.8045


In [7]:
hgb = Pipeline([('pre', preprocessor),
                  ('clf', HistGradientBoostingClassifier(random_state=42, max_iter=300))])

grid_hgb = GridSearchCV(hgb, {
    'clf__learning_rate': [0.05, 0.1],
    'clf__max_depth': [3, 5],
    'clf__min_samples_leaf': [20, 50],
}, cv=5, scoring='accuracy', n_jobs=-1)
grid_hgb.fit(X_train, y_train)

print('Best CV  (train):', round(grid_hgb.best_score_, 4))
print('Best params:', grid_hgb.best_params_)
print('Validation     :', round(accuracy_score(y_val, grid_hgb.predict(X_val)), 4))

Best CV  (train): 0.8343
Best params: {'clf__learning_rate': 0.05, 'clf__max_depth': 3, 'clf__min_samples_leaf': 20}
Validation     : 0.7989


## Other good models you could use

All of these fit this problem well — try the ones already installed first, and
optionally install `xgboost`/`lightgbm`/`catboost` via `uv add`.

| Model/approach | Where it shines on Titanic | Status |
|---|---|---|
| **LogisticRegression** | Simple, stable baseline (~81-83% CV) | installed |
| **RandomForest / ExtraTrees** | Robust to noise, strong default | installed |
| **HistGradientBoosting / GradientBoosting** | Usually the best single learner here (~84% CV) | installed |
| **XGBoost / LightGBM / CatBoost** | Slightly better than sklearn's boosting, needs `pip install` | not installed |
| **SVM (RBF)** | Good after scaling; sensitive to `C`/`gamma` | installed |
| **k-NN** (`model.ipynb`) | Easy but weak on this small/noisy data | installed |
| **MLPClassifier** (small NN) | Rarely beats boosting on 891 rows | installed |
| **Voting/Stacking** | Blend the 2 best models -> often +0.5-1% over the best single | next cell |

**Rules of thumb:**
- Every model goes through the same `preprocessor` so comparisons are fair.
- Pick on **CV mean**, never on the validation split.
- If a model beats a baseline, verify the gap is larger than the CV std — on 5
  folds the spread is typically ~2%, so a 0.5% edge is noise.
- Feature ideas that usually help: extract `Title` from `Name`, keep `Age`/`Fare`
  binned, and encode `Cabin`/`Ticket_code` (already done above).

In [8]:
voting = VotingClassifier([
    ('rf', grid.best_estimator_),
    ('hgb', grid_hgb.best_estimator_),
], voting='soft')

mean, std = cv_score(voting)
print(f'Voting ensemble     CV: {mean:.4f} +/- {std:.4f}')

voting.fit(X_train, y_train)
print('Voting validation     :',
      round(accuracy_score(y_val, voting.predict(X_val)), 4))

Voting ensemble     CV: 0.8339 +/- 0.0309


Voting validation     : 0.8156


## Final submission

Pick the best model by CV mean, retrain on **all** training data (no holdout),
predict the real `test.csv`, and write `../submition/submission.csv` for Kaggle.

In [9]:
final = voting.fit(X, y)

sub = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': final.predict(X_test),
})
sub.to_csv('submission.csv', index=False)

surv = sub['Survived'].mean()
print(f'{len(sub)} predictions written to submission.csv')
print(f'Predicted survival rate: {surv:.1%} (train had {y.mean():.1%})')

418 predictions written to submission.csv
Predicted survival rate: 33.3% (train had 38.4%)
